In [0]:
# Databricks notebook source
# Gold - Squad 3 - ecommerce_clientes_cadastros_mensal
# Fluxo: Silver Delta -> Gold Delta -> SQL Server
#
# Objetivo:
# Criar uma tabela Gold mensal com indicadores de cadastro de clientes:
# - quantidade de novos clientes por mês
# - quantidade acumulada de clientes
# - crescimento mês a mês

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Tabela de origem na camada Silver
SILVER_TABLE = "ecommerce_clientes"
SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"

# Tabela de destino na camada Gold
GOLD_TABLE = "gold_ecommerce_clientes_cadastros_mensal"
GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"

# Chave lógica da Gold
# Como a tabela será mensal, cada linha representa um ano/mês de cadastro
GOLD_KEY_COLUMNS = ["ano_cadastro", "mes_cadastro"]

# Tabela final no SQL Server
SQL_GOLD_TABLE = f"{TARGET_SCHEMA}.{GOLD_TABLE}"

print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("GOLD_KEY_COLUMNS:", GOLD_KEY_COLUMNS)
print("SQL_GOLD_TABLE:", SQL_GOLD_TABLE)

In [0]:
adls_options = get_adls_options()

print("Opções ADLS configuradas.")

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    year,
    month,
    to_date,
    current_timestamp
)

In [0]:
df_silver_clientes = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
)

print("Silver carregada com sucesso.")
print(f"Total de registros na Silver: {df_silver_clientes.count()}")

display(df_silver_clientes.limit(10))

In [0]:
required_columns = [
    "id_cliente",
    "dt_cadastro",
    "ano",
    "mes"
]

missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in df_silver_clientes.columns
]

if missing_columns:
    raise ValueError(f"Colunas obrigatórias ausentes na Silver: {missing_columns}")

print("Todas as colunas obrigatórias estão presentes.")

In [0]:
total_registros = df_silver_clientes.count()

total_clientes_distintos = (
    df_silver_clientes
    .select(countDistinct("id_cliente").alias("total_clientes_distintos"))
    .collect()[0]["total_clientes_distintos"]
)

total_dt_cadastro_nula = (
    df_silver_clientes
    .filter(col("dt_cadastro").isNull())
    .count()
)

print(f"Total de registros: {total_registros}")
print(f"Total de clientes distintos: {total_clientes_distintos}")
print(f"Registros com dt_cadastro nula: {total_dt_cadastro_nula}")

if total_dt_cadastro_nula > 0:
    print("Atenção: existem clientes sem dt_cadastro. Eles não entrarão corretamente na Gold mensal.")
else:
    print("Validação OK: não há dt_cadastro nula.")

In [0]:
from pyspark.sql.functions import (
    date_trunc,
    sum as spark_sum,
    lag,
    round,
    when,
    lit
)

from pyspark.sql.window import Window

In [0]:
df_clientes_base_gold = (
    df_silver_clientes
    .filter(col("dt_cadastro").isNotNull())
    .select(
        col("id_cliente"),
        col("dt_cadastro"),
        year(col("dt_cadastro")).alias("ano_cadastro"),
        month(col("dt_cadastro")).alias("mes_cadastro"),
        to_date(date_trunc("month", col("dt_cadastro"))).alias("data_referencia")
    )
)

display(df_clientes_base_gold.limit(10))

In [0]:
df_gold_mensal_base = (
    df_clientes_base_gold
    .groupBy(
        "ano_cadastro",
        "mes_cadastro",
        "data_referencia"
    )
    .agg(
        countDistinct("id_cliente").alias("qtd_clientes_novos")
    )
)

display(
    df_gold_mensal_base
    .orderBy("data_referencia")
)

In [0]:
window_mensal = Window.orderBy("data_referencia")

df_gold_clientes_cadastros_mensal = (
    df_gold_mensal_base
    .withColumn(
        "qtd_clientes_acumulado",
        spark_sum("qtd_clientes_novos").over(window_mensal)
    )
    .withColumn(
        "qtd_clientes_novos_mes_anterior",
        lag("qtd_clientes_novos").over(window_mensal)
    )
    .withColumn(
        "crescimento_mom_percentual",
        when(
            col("qtd_clientes_novos_mes_anterior").isNull(),
            lit(None)
        ).when(
            col("qtd_clientes_novos_mes_anterior") == 0,
            lit(None)
        ).otherwise(
            round(
                (
                    (col("qtd_clientes_novos") - col("qtd_clientes_novos_mes_anterior"))
                    / col("qtd_clientes_novos_mes_anterior")
                ) * 100,
                2
            )
        )
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
    .select(
        "ano_cadastro",
        "mes_cadastro",
        "data_referencia",
        "qtd_clientes_novos",
        "qtd_clientes_novos_mes_anterior",
        "crescimento_mom_percentual",
        "qtd_clientes_acumulado",
        "gold_processed_at"
    )
)

display(
    df_gold_clientes_cadastros_mensal
    .orderBy("data_referencia")
)

In [0]:
total_linhas_gold = df_gold_clientes_cadastros_mensal.count()

total_chaves_distintas = (
    df_gold_clientes_cadastros_mensal
    .select("ano_cadastro", "mes_cadastro")
    .distinct()
    .count()
)

print(f"Total de linhas na Gold: {total_linhas_gold}")
print(f"Total de chaves ano/mês distintas: {total_chaves_distintas}")

if total_linhas_gold != total_chaves_distintas:
    raise ValueError("Erro: existem meses duplicados na Gold.")

print("Validação OK: a Gold possui apenas uma linha por mês.")

In [0]:
total_clientes_silver_validos = (
    df_silver_clientes
    .filter(col("dt_cadastro").isNotNull())
    .select("id_cliente")
    .distinct()
    .count()
)

total_clientes_gold = (
    df_gold_clientes_cadastros_mensal
    .agg(
        spark_sum("qtd_clientes_novos").alias("total_clientes_gold")
    )
    .collect()[0]["total_clientes_gold"]
)

print(f"Total de clientes válidos na Silver: {total_clientes_silver_validos}")
print(f"Total de clientes somados na Gold: {total_clientes_gold}")

if total_clientes_silver_validos != total_clientes_gold:
    raise ValueError("Erro: total de clientes da Gold não bate com a Silver.")

print("Validação OK: total da Gold bate com a Silver.")

In [0]:
df_gold_clientes_cadastros_mensal.printSchema()

display(
    df_gold_clientes_cadastros_mensal
    .orderBy("data_referencia")
)

In [0]:
from delta.tables import DeltaTable

In [0]:
try:
    df_gold_existente = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(GOLD_PATH)
    )

    total_registros_gold_existente = df_gold_existente.count()

    gold_delta_exists = True

    print("Tabela Gold Delta já existe.")
    print(f"Registros atuais na Gold Delta: {total_registros_gold_existente}")

except Exception as error:
    gold_delta_exists = False

    print("Tabela Gold Delta ainda não existe.")
    print("Na primeira execução, ela será criada com overwrite.")

In [0]:
if not gold_delta_exists:
    (
        df_gold_clientes_cadastros_mensal
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .options(**adls_options)
        .save(GOLD_PATH)
    )

    print("Tabela Gold Delta criada com sucesso.")

else:
    df_gold_existente = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(GOLD_PATH)
    )

    df_chaves_novas = (
        df_gold_clientes_cadastros_mensal
        .select("ano_cadastro", "mes_cadastro")
        .distinct()
    )

    df_gold_existente_sem_meses_atualizados = (
        df_gold_existente
        .join(
            df_chaves_novas,
            on=["ano_cadastro", "mes_cadastro"],
            how="left_anti"
        )
    )

    df_gold_upsert = (
        df_gold_existente_sem_meses_atualizados
        .unionByName(df_gold_clientes_cadastros_mensal)
    )

    (
        df_gold_upsert
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .options(**adls_options)
        .save(GOLD_PATH)
    )

    print("Tabela Gold Delta atualizada com upsert manual.")

In [0]:
df_gold_delta_validacao = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

print("Gold Delta carregada com sucesso.")
print(f"Total de registros na Gold Delta: {df_gold_delta_validacao.count()}")

display(
    df_gold_delta_validacao
    .orderBy("data_referencia")
)

In [0]:
df_gold_chaves_duplicadas = (
    df_gold_delta_validacao
    .groupBy("ano_cadastro", "mes_cadastro")
    .count()
    .filter(col("count") > 1)
)

total_chaves_duplicadas = df_gold_chaves_duplicadas.count()

print(f"Total de chaves duplicadas na Gold Delta: {total_chaves_duplicadas}")

if total_chaves_duplicadas > 0:
    display(df_gold_chaves_duplicadas)
    raise ValueError("Erro: existem chaves duplicadas na Gold Delta.")

print("Validação OK: não existem meses duplicados na Gold Delta.")

In [0]:
total_gold_memoria = df_gold_clientes_cadastros_mensal.count()
total_gold_delta = df_gold_delta_validacao.count()

print(f"Total de linhas na Gold em memória: {total_gold_memoria}")
print(f"Total de linhas na Gold Delta: {total_gold_delta}")

if total_gold_memoria != total_gold_delta:
    raise ValueError("Erro: total de linhas da Gold Delta não bate com a Gold em memória.")

print("Validação OK: Gold Delta bate com a Gold em memória.")

In [0]:
SQL_FINAL_TABLE = f"{TARGET_SCHEMA}.{GOLD_TABLE}"
SQL_STAGING_TABLE = f"{TARGET_SCHEMA}.stg_{GOLD_TABLE}"

print("Tabela final SQL Server:", SQL_FINAL_TABLE)
print("Tabela staging SQL Server:", SQL_STAGING_TABLE)

In [0]:
df_gold_sql = (
    df_gold_delta_validacao
    .select(
        "ano_cadastro",
        "mes_cadastro",
        "data_referencia",
        "qtd_clientes_novos",
        "qtd_clientes_novos_mes_anterior",
        "crescimento_mom_percentual",
        "qtd_clientes_acumulado",
        "gold_processed_at"
    )
)

print(f"Total de registros preparados para SQL Server: {df_gold_sql.count()}")

display(
    df_gold_sql
    .orderBy("data_referencia")
)

In [0]:
validation_result = validate_key_columns(
    df_gold_sql,
    ["ano_cadastro", "mes_cadastro"]
)

print(validation_result["message"])
print("Chave validada:", validation_result["key_columns"])
print("Registros duplicados:", validation_result["duplicated_rows"])

In [0]:
if TARGET_SCHEMA != "squad3":
    raise ValueError(f"Schema inesperado: {TARGET_SCHEMA}. Esperado: squad3.")

if not SQL_STAGING_TABLE.startswith("squad3.stg_"):
    raise ValueError(f"Tabela staging inesperada: {SQL_STAGING_TABLE}")

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=SQL_STAGING_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print("Tabela staging gravada com sucesso no SQL Server.")
print(f"Staging criada/atualizada: {SQL_STAGING_TABLE}")

In [0]:
df_sql_staging_validacao = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=SQL_STAGING_TABLE,
    sql_port=SQL_PORT
)

print("Staging lida com sucesso do SQL Server.")
print(f"Total de registros na staging SQL Server: {df_sql_staging_validacao.count()}")

display(
    df_sql_staging_validacao
    .orderBy("data_referencia")
)

In [0]:
total_gold_delta = df_gold_delta_validacao.count()
total_sql_staging = df_sql_staging_validacao.count()

print(f"Total de registros na Gold Delta: {total_gold_delta}")
print(f"Total de registros na staging SQL Server: {total_sql_staging}")

if total_gold_delta != total_sql_staging:
    raise ValueError("Erro: total da staging SQL Server não bate com a Gold Delta.")

print("Validação OK: staging SQL Server bate com a Gold Delta.")

In [0]:
df_staging_chaves_duplicadas = (
    df_sql_staging_validacao
    .groupBy("ano_cadastro", "mes_cadastro")
    .count()
    .filter(col("count") > 1)
)

total_duplicadas_staging = df_staging_chaves_duplicadas.count()

print(f"Total de chaves duplicadas na staging SQL Server: {total_duplicadas_staging}")

if total_duplicadas_staging > 0:
    display(df_staging_chaves_duplicadas)
    raise ValueError("Erro: existem chaves duplicadas na staging SQL Server.")

print("Validação OK: staging SQL Server sem duplicidades.")

In [0]:
try:
    df_sql_final_existente = read_sql_table(
        spark=spark,
        sql_host=SQL_HOST,
        sql_database=SQL_DATABASE,
        sql_username=SQL_USERNAME,
        sql_password=SQL_PASSWORD,
        table_name=SQL_FINAL_TABLE,
        sql_port=SQL_PORT
    )

    total_sql_final_existente = df_sql_final_existente.count()
    sql_final_exists = True

    print("Tabela final já existe no SQL Server.")
    print(f"Registros atuais na tabela final: {total_sql_final_existente}")

except Exception:
    sql_final_exists = False

    print("Tabela final ainda não existe no SQL Server.")
    print("Na primeira execução, ela será criada.")

In [0]:
if not sql_final_exists:
    write_sql_table(
        df=df_sql_staging_validacao,
        sql_host=SQL_HOST,
        sql_database=SQL_DATABASE,
        sql_username=SQL_USERNAME,
        sql_password=SQL_PASSWORD,
        table_name=SQL_FINAL_TABLE,
        mode="overwrite",
        sql_port=SQL_PORT
    )

    print("Tabela final criada com sucesso no SQL Server.")

else:
    df_chaves_staging = (
        df_sql_staging_validacao
        .select("ano_cadastro", "mes_cadastro")
        .distinct()
    )

    df_sql_final_sem_meses_atualizados = (
        df_sql_final_existente
        .join(
            df_chaves_staging,
            on=["ano_cadastro", "mes_cadastro"],
            how="left_anti"
        )
    )

    df_sql_final_upsert = (
        df_sql_final_sem_meses_atualizados
        .unionByName(df_sql_staging_validacao)
    )

    write_sql_table(
        df=df_sql_final_upsert,
        sql_host=SQL_HOST,
        sql_database=SQL_DATABASE,
        sql_username=SQL_USERNAME,
        sql_password=SQL_PASSWORD,
        table_name=SQL_FINAL_TABLE,
        mode="overwrite",
        sql_port=SQL_PORT
    )

    print("Tabela final atualizada com upsert manual no SQL Server.")

In [0]:
df_sql_final_validacao = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=SQL_FINAL_TABLE,
    sql_port=SQL_PORT
)

print("Tabela final lida com sucesso do SQL Server.")
print(f"Total de registros na tabela final: {df_sql_final_validacao.count()}")

display(
    df_sql_final_validacao
    .orderBy("data_referencia")
)

In [0]:
total_sql_staging = df_sql_staging_validacao.count()
total_sql_final = df_sql_final_validacao.count()

print(f"Total de registros na staging: {total_sql_staging}")
print(f"Total de registros na tabela final: {total_sql_final}")

if total_sql_final < total_sql_staging:
    raise ValueError("Erro: tabela final possui menos registros que a staging.")

df_final_chaves_duplicadas = (
    df_sql_final_validacao
    .groupBy("ano_cadastro", "mes_cadastro")
    .count()
    .filter(col("count") > 1)
)

total_final_duplicadas = df_final_chaves_duplicadas.count()

print(f"Total de chaves duplicadas na tabela final: {total_final_duplicadas}")

if total_final_duplicadas > 0:
    display(df_final_chaves_duplicadas)
    raise ValueError("Erro: existem chaves duplicadas na tabela final.")

print("Validação OK: tabela final publicada corretamente no SQL Server.")

In [0]:
print("Processamento Gold concluído com sucesso.")
print(f"Tabela Gold Delta: {GOLD_PATH}")
print(f"Tabela staging SQL Server: {SQL_STAGING_TABLE}")
print(f"Tabela final SQL Server: {SQL_FINAL_TABLE}")

print(f"Total de registros na Gold Delta: {df_gold_delta_validacao.count()}")
print(f"Total de registros na staging SQL Server: {df_sql_staging_validacao.count()}")
print(f"Total de registros na tabela final SQL Server: {df_sql_final_validacao.count()}")